In [0]:
catalog = "cine_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"


# Checagem de Data Quality.

In [0]:
from pyspark.sql import functions as F ,Row
from datetime import datetime

dq_results =[]
def dq_check(table_name: str, check_name: str, df,condition ):
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed==0
    dq_results.append(Row(table_name = table_name, check_name=check_name,
                           total_rows = total, failed_rows = failed,
                            passed=passed,checked_at = datetime.now()))
    status= "pass" if passed else "Failed"
    print(f"[{status}] {table_name} | {check_name} |{failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

In [0]:
df_bronze_info = spark.table(f"{bronze_schema}.info")

In [0]:
dq_check("bronze.info","id não nulo", df_bronze_info,F.col("id").isNotNull())

dq_check_unique("bronze.info", "id único", df_bronze_info, ["id"])

In [0]:
df_silver_info = (
    df_bronze_info \
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title","titulo")
    .withColumnRenamed("original_title","titulo_original")
    .withColumnRenamed("release_date","data_lancamento")
    .withColumnRenamed('runtime','duracao_minutos')
    .withColumnRenamed('original_language','idioma_original')
    .withColumnRenamed('status','status_filme')
    .withColumnRenamed('overview','sinopse')
    .withColumnRenamed('tagline','frase_divulgacao')

)

In [0]:
from pyspark.sql.window import Window
def drop_and_keep(column_name:str, ingestion_column_name:str):
    window = Window.partitionBy(column_name).orderBy(
        F.col(ingestion_column_name).desc_nulls_last(),
    )
    return window
window = drop_and_keep('status_filme',"ingestion_datetime")

df_ranked_info =( 
                 df_silver_info
                 .withColumn("_row_num",F.row_number().over(window))
)

In [0]:

status_mapping = {
    "released": "Lançado",
    "post production": "Pós-Produção",
    "in production": "Em Produção",
    "planned": "Planejado",
    "rumored": "Rumores",
    "canceled": "Cancelado",
}

# Lowercase, replace punctuation/hyphens/noise with spaces,
# collapse repeated spaces, and trim.
normalized_status = F.trim(
    F.regexp_replace(
        F.lower(F.col("status_filme")),
        r"[^a-z]+",
        " "
    )
)

mapping_expr = F.create_map(
    *[
        F.lit(value)
        for pair in status_mapping.items()
        for value in pair
    ]
)

df_silver_info = df_silver_info.withColumn(
    "status_filme",
    F.coalesce(
        F.element_at(mapping_expr, normalized_status),
        F.lit("Não Informado")
    )
)

In [0]:
df_silver_info.select("status_filme").groupBy("status_filme").count().show(15)

In [0]:

dq_check("silver.info","titulo nulo", df_silver_info,F.col("titulo").isNotNull())
dq_check("silver.info","id nulo", df_silver_info,F.col("id").isNotNull())
dq_check("silver.info","data lançamento", df_silver_info,F.col("data_lancamento").isNotNull())


In [0]:
from pyspark.sql import functions as F

# Ajuste esta lista aos formatos presentes na origem.
formatos = [
    "yyyy-MM-dd",
    "yyyy/MM/dd",
    "dd/MM/yyyy",
    "dd-MM-yyyy",
    "MM-yyyy",
    "yyyyMMdd",
    "yyyy-MM-dd HH:mm:ss",
    "dd/MM/yyyy HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss[.SSSSSSSSS]",
]

data_texto = F.trim(F.col("data_lancamento").cast("string"))

data_convertida = F.coalesce(
    *[
        F.try_to_timestamp(data_texto, F.lit(formato))
        for formato in formatos
    ]
).cast("date")

df_silver_info = (
    df_silver_info
    .withColumn("data_lancamento_original", F.col("data_lancamento"))
    .withColumn("data_lancamento", data_convertida)
)


In [0]:
(
    df_silver_info
    .filter(
        F.col("data_lancamento_original").isNotNull()
        & F.col("data_lancamento").isNull()
    )
    .select("data_lancamento_original")
    .distinct()
    .show(50, truncate=False)
)

In [0]:
df_silver_info = df_silver_info.withColumn(
    "ano_lancamento",
    F.year("data_lancamento")
)


In [0]:
df_bronze_info = spark.table(f"{bronze_schema}.info")